In [ ]:
# 验证混合方法的合理性

import numpy as np
from scipy.linalg import toeplitz
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import copy
import math

rank = np.zeros(100)
results = [0,0,0]
with open('output.csv', 'w') as f:
    f.write('improvement,smallest,submodular\n')  # 可选：写入列名
#result = 0

for seed in tqdm(range(100)):
    # 1. 参数设置
    p = 100  # 科目总数
    m = 10 # 考试科目数
    n_samples = 1000  
    J = 10000
    np.random.seed(seed)

    # 2. 生成均值（70-90分）
    mu = np.random.uniform(70, 90, size=p)

    # 3. 生成协方差矩阵
    group_sizes = [15, 10, 20, 10, 15, 10, 10, 10]  # 8个科目组
    Sigma = np.zeros((p, p))
    start = 0
    for size in group_sizes:
        end = start + size
        block_corr = np.random.uniform(0.6, 1)
        block = np.ones((size, size)) * block_corr
        np.fill_diagonal(block, 1)
        Sigma[start:end, start:end] = block 
        start = end

    # 结合方差
    variances = np.random.uniform(25, 49, size=p)
    D = np.diag(np.sqrt(variances))
    Sigma = D @ Sigma @ D

    # 4. 生成数据并截断
    X = np.random.multivariate_normal(mu, Sigma, size=n_samples)
    X = np.clip(X, 0, 100)

    mu = mu
    std = np.sqrt(np.diag(Sigma))
    R = np.corrcoef(X[:, :].T)

    def Emin2(id1,id2): #计算两个正态随机变量的最小值期望
        mu1 = mu[id1]
        mu2 = mu[id2]
        std1 = std[id1]
        std2 = std[id2]
        rou = R[id1,id2]
        theta = math.sqrt(std1*std1+std2*std2-2*rou*std1*std2)+1e-6
        result = mu1*norm.cdf((mu2-mu1)/theta)+mu2*norm.cdf((mu1-mu2)/theta)-theta*norm.pdf((mu2-mu1)/theta)
        return result
#-----------------------------------------------------------------------------------------
    choosen_index = [] #选定科目
    remain_index = list(range(0,p)) #剩余科目
    choosen_index.append(int(np.argmin(mu)))
    remain_index.remove(np.argmin(mu))

    for j in range(m-1):
        best_improvement = -1 #全局最优
        best_id = -1
        for id1 in remain_index:
            improvement_min = 100 #局部最小
            for id2 in choosen_index:
                temp = mu[id2]-Emin2(id1,id2)
                if temp < improvement_min:
                    improvement_min = temp
            if improvement_min > best_improvement:
                best_improvement = improvement_min
                best_id = id1
        choosen_index.append(best_id)
        remain_index.remove(best_id)

    sub_Sigma = Sigma[np.ix_(choosen_index, choosen_index)]
    B = np.linalg.cholesky(sub_Sigma) #对协方差矩阵进行cholesky分解
#-----------------------------------------------------------------------------------------
    indices_smallest = np.argsort(mu)[:m] #直接取最小值
    sub_Sigma = Sigma[np.ix_(indices_smallest, indices_smallest)]
    L = np.linalg.cholesky(sub_Sigma) 
#-----------------------------------------------------------------------------------------
    submodular_index = [choosen_index[0],choosen_index[1]]
    Gmin = 100
    bestid = -1
    bestimp = -1
    for t in range(m-2):
        imp = []
        for i in range(p):
            improvement_min = 100
            if i not in submodular_index:
                copy_index = copy.deepcopy(submodular_index)
                copy_index.append(i)
                sub_Sigma = Sigma[np.ix_(copy_index, copy_index)]
                sub_B = np.linalg.cholesky(sub_Sigma) #对协方差矩阵进行cholesky分解
                G_cul = 0
                for j in range(J):
                    z = np.random.randn(len(copy_index))
                    temp = mu[copy_index] + sub_B @ z
                    G_cul += np.min(temp)
                G_temp = G_cul/J
                for id2 in submodular_index:
                    temp = mu[id2]-Emin2(i,id2)
                    if temp < improvement_min:
                        improvement_min = temp
                imp.append(improvement_min)
                if G_temp < Gmin:
                    Gmin = G_temp
                    bestid = i
                    bestimp = improvement_min
        submodular_index.append(bestid)
        imp.sort()
        targetindex = imp.index(bestimp)
        rank[targetindex]+=1
        Gmin = 100
        bestimp = -1

    sub_Sigma = Sigma[np.ix_(submodular_index, submodular_index)]
    U = np.linalg.cholesky(sub_Sigma)
#-----------------------------------------------------------------------------------------
    G_cul = 0
    for j in range(J):
        z = np.random.randn(m)
        temp = mu[choosen_index] + B @ z
        G_cul += np.min(temp)
    G1 = G_cul/J

    G_cul = 0
    for j in range(J):
        z = np.random.randn(m)
        temp = mu[indices_smallest] + L @ z
        G_cul += np.min(temp)
    G2 = G_cul/J

    G_cul = 0
    for j in range(J):
        z = np.random.randn(m)
        temp = mu[submodular_index] + U @ z
        G_cul += np.min(temp)
    G3 = G_cul/J
    G = np.array([G1,G2,G3])
    results[np.argmin(G)]+=1

    with open('output.csv', 'a') as f: 
        line = ','.join(map(str, G)) + '\n'  # 转为逗号分隔的字符串
        f.write(line)

print(results)



100%|██████████| 100/100 [54:16<00:00, 32.57s/it]

[29, 4, 67]


In [26]:
imprank = np.zeros(p-2)
for i in range(p-2):
    imprank[i] = rank[p-3-i]
imprank

array([70., 81., 88., 93., 79., 80., 70., 61., 43., 32., 19., 11., 21.,
        8.,  5.,  7.,  9.,  9.,  3.,  2.,  3.,  0.,  1.,  1.,  0.,  0.,
        0.,  1.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  1.])

In [27]:
np.savetxt('rank.txt', imprank, fmt='%d')     

In [30]:
np.sum(imprank[:10])/800

np.float64(0.87125)

In [28]:
import numpy as np
import pandas as pd
data = pd.read_csv('output.csv')
improvement = np.mean(data['improvement'].to_numpy())
smallest = np.mean(data['smallest'].to_numpy())
submodular = np.mean(data['submodular'].to_numpy())
print('improvement:%f\nsmallest:%f\nsubmodular:%f\n'%(improvement,smallest,submodular))

improvement:62.164691
smallest:62.488868
submodular:62.109964

